<a href="https://colab.research.google.com/github/Shakeel1111-creator/RAG-Based-University-Student-Support-Chatbot-using-ChromaDB-and-Qwen-2.5/blob/main/RAG_Based_University_Student_Support_Chatbot_using_ChromaDB_and_Qwen_2_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install -q torch accelerate sentence-transformers chromadb
import torch
import chromadb
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
embedding_model=SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)
model_name="Qwen/Qwen2.5-0.5B-Instruct"

tokenizer=AutoTokenizer.from_pretrained(model_name)
llm = AutoModelForCausalLM.from_pretrained(model_name)
print("Models loaded successfully")

documents = [
    {
        "id": "doc1",
        "text": """
        Students must maintain at least 75% attendance
        to be eligible for semester examinations.
        Attendance records are maintained by the respective
        departments. Students with attendance below 75%
        may not be permitted to attend examinations.
        """
    },
    {
        "id": "doc2",
        "text": """
        The university library operates from 9 AM to 6 PM,
        Monday to Saturday. Students can borrow up to
        three books at a time. Books can be borrowed for
        a maximum period of 14 days.
        Students must return books before the due date.
        """
    },
    {
        "id": "doc3",
        "text": """
        Students can apply for internships through the
        university placement cell. Students must submit
        their updated resume. Internship opportunities
        are shared through the university placement portal.
        The placement cell coordinates with companies
        for internship opportunities.
        """
    },
    {
        "id": "doc4",
        "text": """
        Students must register for semester examinations
        through the university portal. Examination fees
        must be paid before the registration deadline.
        Hall tickets can be downloaded from the student
        portal after successful registration.
        """
    }
]

print("Total documents:", len(documents))

chunk_size=150
overlap=30
step=chunk_size-overlap

chunks=[]

for document in documents:
  text=document["text"]
  document_id=document["id"]
  chunk_number=1

  for start in range(0,len(text),step):
    chunk=text[start:start+chunk_size]

    chunks.append({
        "text":chunk,
        "metadata":{
            "document_id":document_id,
            "chunk_number":chunk_number
        }
    })
    chunk_number+=1

    if start+chunk_size>=len(text):
      break

print("Total chunks: ",len(chunks))

for chunk in chunks:
  print("\n chunk: ",chunk["text"])
  print("Metadata: ",chunk["metadata"])

chunk_text=[
    chunk["text"] for chunk in chunks
]

chunk_embeddings=embedding_model.encode(
    chunk_text,normalize_embeddings=True
)

print("Embeddings shape: ",chunk_embeddings.shape)

client=chromadb.Client()

collection=client.get_or_create_collection(
    name="college_campus"
)

chunk_ids = [
    f"{chunk['metadata']['document_id']}_chunk_{chunk['metadata']['chunk_number']}"
    for chunk in chunks
]

chunk_metadatas=[ chunk["metadata"] for chunk in chunks
]

collection.upsert(ids=chunk_ids,documents=chunk_text,embeddings=chunk_embeddings.tolist(),metadatas=chunk_metadatas)

print("Total chunks stored: ",collection.count())

query="what is the minimum attendance requirement?"

print("\nStudent question: ",query)

query_embedding=embedding_model.encode([query],normalize_embeddings=True)
print("query embeddings shape: ",query_embedding.shape)

results=collection.query(query_embeddings=query_embedding.tolist(),n_results=3)

retrieved_documents=results["documents"][0]
retrieved_metadatas=results["metadatas"][0]

retrieved_distances=results["distances"][0]

for document,metadata,distance in zip(retrieved_documents,retrieved_metadatas,retrieved_distances):
  print("\nDocument: ",document)
  print("Metadata: ",metadata)
  print("Distance: ",distance)

context="\n".join(retrieved_documents)

print("context:",context)


messages = [
    {
        "role": "system",
        "content": """
        You are a helpful university student support assistant.

        Answer using only the information in the provided context.

        If the answer is not available in the context,
        say that you do not have enough information.

        Do not invent information.
        """
    },
    {
        "role": "user",
        "content": f"""
        Context:
        {context}

        Question:
        {query}
        """
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

print("prompt: ",prompt)
with torch.no_grad():
  output_ids=llm.generate(**inputs,max_new_tokens=150,do_sample=False)


input_length=inputs["input_ids"].shape[1]

new_tokens=output_ids[0][input_length: ]

answer=tokenizer.decode(new_tokens,skip_special_tokens=True)

print("AI answer: ",answer)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (14 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Models loaded successfully
Total documents: 4
Total chunks:  12

 chunk:  
        Students must maintain at least 75% attendance
        to be eligible for semester examinations.
        Attendance records are maintained by
Metadata:  {'document_id': 'doc1', 'chunk_number': 1}

 chunk:  ance records are maintained by the respective
        departments. Students with attendance below 75%
        may not be permitted to attend examinati
Metadata:  {'document_id': 'doc1', 'chunk_number': 2}

 chunk:   permitted to attend examinations.
        
Metadata:  {'document_id': 'doc1', 'chunk_number': 3}

 chunk:  
        The university library operates from 9 AM to 6 PM,
        Monday to Saturday. Students can borrow up to
        three books at a time. Books
Metadata:  {'document_id': 'doc2', 'chunk_number': 1}

 chunk:    three books at a time. Books can be borrowed for
        a maximum period of 14 days.
        Students must return books before the due date.
      
Metadata:  {'document_